# Week 9: Correlation & Regression — Applied Statistics with Python (2026)

So far we've tested whether groups differ. This week we move to **relationships**: how does one variable relate to another? We start with **correlation** (measuring the strength and direction of association) and progress to **regression** (modeling and predicting one variable from others). These are among the most widely used statistical techniques in science, business, and engineering.

| # | Topic | Key Concepts |
|---|-------|--------------|
| 1 | Scatter Plots & Relationships | Visualizing bivariate data |
| 2 | Pearson Correlation | Linear association, r, significance test |
| 3 | Spearman Rank Correlation | Non-parametric, monotonic relationships |
| 4 | Correlation Pitfalls | ≠ causation, outliers, Simpson's paradox |
| 5 | Simple Linear Regression | OLS, slope & intercept, R² |
| 6 | Regression with statsmodels | OLS summary, coefficient interpretation |
| 7 | Regression Diagnostics | Residual analysis, assumption checking |
| 8 | Multiple Linear Regression | Multiple predictors |
| 9 | Categorical Predictors | Dummy variables |
| 10 | Model Comparison | Adjusted R², AIC, BIC |
| 11 | Practical Example | Complete regression pipeline |
| 12 | Summary | Key functions and concepts |
| 13 | Homework | Practice exercises |

### Import all necessary libraries for this week's analysis.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Consistent style
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')
np.random.seed(42)

print('All libraries loaded successfully!')

---
## 1. Scatter Plots & Types of Relationships

Before computing any statistics, **always visualize** the relationship between two variables with a scatter plot. Relationships can be:

| Type | Description | Example |
|------|-------------|---------|
| **Linear positive** | As x↑, y↑ proportionally | Study hours → exam score |
| **Linear negative** | As x↑, y↓ proportionally | Price → demand |
| **Non-linear** | Curved relationship | Age → income (rises then falls) |
| **No relationship** | No pattern | Shoe size → IQ |

**Anscombe's Quartet** famously shows that summary statistics can be identical for wildly different patterns — always plot first!

Generate four different types of bivariate relationships and visualize them to build intuition for scatter plots.

In [ ]:
np.random.seed(42)
n = 100
x = np.random.uniform(0, 10, n)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# 1. Strong positive linear
y1 = 2 * x + 3 + np.random.normal(0, 2, n)
axes[0, 0].scatter(x, y1, alpha=0.6, color='steelblue')
axes[0, 0].set_title(f'Strong Positive Linear\nr = {np.corrcoef(x, y1)[0,1]:.3f}', fontsize=13)

# 2. Moderate negative linear
y2 = -1.5 * x + 20 + np.random.normal(0, 4, n)
axes[0, 1].scatter(x, y2, alpha=0.6, color='coral')
axes[0, 1].set_title(f'Moderate Negative Linear\nr = {np.corrcoef(x, y2)[0,1]:.3f}', fontsize=13)

# 3. Non-linear (quadratic)
y3 = (x - 5)**2 + np.random.normal(0, 2, n)
axes[1, 0].scatter(x, y3, alpha=0.6, color='seagreen')
axes[1, 0].set_title(f'Non-Linear (Quadratic)\nr = {np.corrcoef(x, y3)[0,1]:.3f} (misleading!)', fontsize=13)

# 4. No relationship
y4 = np.random.normal(10, 3, n)
axes[1, 1].scatter(x, y4, alpha=0.6, color='purple')
axes[1, 1].set_title(f'No Relationship\nr = {np.corrcoef(x, y4)[0,1]:.3f}', fontsize=13)

for ax in axes.flat:
    ax.set_xlabel('X', fontsize=11)
    ax.set_ylabel('Y', fontsize=11)

plt.suptitle('Types of Bivariate Relationships', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

Demonstrate Anscombe's Quartet: four datasets with identical summary statistics but very different patterns.

In [ ]:
# Anscombe's Quartet — built into seaborn
anscombe = sns.load_dataset('anscombe')

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for ax, dataset in zip(axes.flat, ['I', 'II', 'III', 'IV']):
    data = anscombe[anscombe['dataset'] == dataset]
    ax.scatter(data['x'], data['y'], s=60, color='steelblue', alpha=0.7)
    
    # Fit and plot regression line
    slope, intercept = np.polyfit(data['x'], data['y'], 1)
    x_line = np.linspace(data['x'].min(), data['x'].max(), 100)
    ax.plot(x_line, slope * x_line + intercept, 'r-', linewidth=2)
    
    r = np.corrcoef(data['x'], data['y'])[0, 1]
    ax.set_title(f'Dataset {dataset}: r = {r:.3f}, mean(x) = {data["x"].mean():.1f}, '
                 f'mean(y) = {data["y"].mean():.2f}', fontsize=11)

plt.suptitle("Anscombe's Quartet: Same Statistics, Different Data!", 
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()
print('All four datasets have nearly identical r, mean, variance, and regression line.')
print('→ ALWAYS visualize your data!')

---
## 2. Pearson Correlation Coefficient

The **Pearson correlation coefficient** (r) measures the **strength and direction** of the **linear** relationship between two continuous variables.

$$r = \frac{\sum(x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum(x_i - \bar{x})^2 \cdot \sum(y_i - \bar{y})^2}}$$

| r value | Interpretation |
|---------|---------------|
| +1.0 | Perfect positive linear |
| +0.7 to +1.0 | Strong positive |
| +0.3 to +0.7 | Moderate positive |
| 0 to +0.3 | Weak positive |
| 0 | No linear relationship |
| Negative | Mirror of above |

**Assumptions:**
1. Both variables are continuous
2. Linear relationship
3. No extreme outliers
4. Approximately normal (for significance test)

### 2.1 Computing Pearson's r

Compute the Pearson correlation between study hours and exam scores using multiple methods.

Calculate Pearson's r manually, with NumPy, with SciPy (which also gives p-value), and with Pandas.

In [ ]:
# Generate study data
np.random.seed(42)
n = 50
study_hours = np.random.uniform(1, 10, n)
exam_scores = 40 + 5 * study_hours + np.random.normal(0, 8, n)  # True relationship + noise
exam_scores = exam_scores.clip(0, 100)

# Method 1: Manual calculation
x_dev = study_hours - study_hours.mean()
y_dev = exam_scores - exam_scores.mean()
r_manual = np.sum(x_dev * y_dev) / np.sqrt(np.sum(x_dev**2) * np.sum(y_dev**2))

# Method 2: NumPy
r_numpy = np.corrcoef(study_hours, exam_scores)[0, 1]

# Method 3: SciPy (gives p-value too!)
r_scipy, p_value = stats.pearsonr(study_hours, exam_scores)

# Method 4: Pandas
df = pd.DataFrame({'hours': study_hours, 'score': exam_scores})
r_pandas = df['hours'].corr(df['score'])

print(f'Pearson r (manual):  {r_manual:.6f}')
print(f'Pearson r (NumPy):   {r_numpy:.6f}')
print(f'Pearson r (SciPy):   {r_scipy:.6f}')
print(f'Pearson r (Pandas):  {r_pandas:.6f}')
print(f'\nP-value: {p_value:.6f}')
print(f'Significant at α=0.05? {"Yes" if p_value < 0.05 else "No"}')
print(f'\nInterpretation: {"Strong" if abs(r_scipy) > 0.7 else "Moderate" if abs(r_scipy) > 0.3 else "Weak"} '
      f'{"positive" if r_scipy > 0 else "negative"} correlation')

### 2.2 Visualizing Correlation

Create publication-quality scatter plots with regression lines and annotated correlation statistics.

Plot the study hours vs exam scores relationship with a regression line, confidence band, and correlation annotation.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Scatter with regression line (seaborn)
ax = axes[0]
sns.regplot(x='hours', y='score', data=df, ax=ax, color='steelblue',
            scatter_kws={'alpha': 0.6, 's': 50},
            line_kws={'color': 'red', 'linewidth': 2})
ax.set_xlabel('Study Hours', fontsize=12)
ax.set_ylabel('Exam Score', fontsize=12)
ax.set_title(f'Study Hours vs Exam Score\nr = {r_scipy:.3f}, p = {p_value:.4f}', fontsize=13)

# 2. What different r values look like
ax = axes[1]
r_targets = [0.0, 0.3, 0.6, 0.9]
colors = ['gray', 'steelblue', 'coral', 'seagreen']
for r_target, color in zip(r_targets, colors):
    # Generate data with specific correlation
    x_temp = np.random.normal(0, 1, 100)
    y_temp = r_target * x_temp + np.sqrt(1 - r_target**2) * np.random.normal(0, 1, 100)
    ax.scatter(x_temp, y_temp, alpha=0.3, s=20, color=color, label=f'r = {r_target}')

ax.set_xlabel('X', fontsize=12)
ax.set_ylabel('Y', fontsize=12)
ax.set_title('What Different r Values Look Like', fontsize=13)
ax.legend(fontsize=11, markerscale=2)

plt.tight_layout()
plt.show()

### 2.3 Correlation Matrix

When we have multiple variables, we compute a **correlation matrix** showing all pairwise Pearson correlations.

Build a multi-variable dataset and compute the correlation matrix with a heatmap visualization.

In [ ]:
# Multi-variable student dataset
np.random.seed(42)
n = 200
study_h = np.random.uniform(1, 10, n)
sleep_h = np.random.normal(7, 1.5, n).clip(3, 12)
attendance = (60 + 4 * study_h + np.random.normal(0, 10, n)).clip(0, 100)
gpa = (1.5 + 0.25 * study_h + 0.1 * sleep_h + 0.01 * attendance 
       + np.random.normal(0, 0.3, n)).clip(0, 4.0)

students = pd.DataFrame({
    'Study Hours': study_h,
    'Sleep Hours': sleep_h,
    'Attendance %': attendance,
    'GPA': gpa
})

# Correlation matrix
corr_matrix = students.corr()
print('Correlation Matrix:')
print(corr_matrix.round(3))

# Heatmap
fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)  # Upper triangle mask
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='RdBu_r', center=0,
            mask=mask, square=True, ax=ax, vmin=-1, vmax=1,
            cbar_kws={'label': 'Pearson r'})
ax.set_title('Correlation Matrix Heatmap', fontsize=14)
plt.tight_layout()
plt.show()

Use a pairplot to visualize all pairwise relationships simultaneously.

In [ ]:
# Pairplot: all pairwise scatter plots + distributions
g = sns.pairplot(students, diag_kind='kde', plot_kws={'alpha': 0.4, 's': 20},
                 diag_kws={'fill': True})
g.fig.suptitle('Pairwise Relationships', fontsize=14, y=1.01)
plt.show()

---
## 3. Spearman Rank Correlation

**Spearman's ρ** (rho) is a non-parametric measure of **monotonic** (not necessarily linear) association. It works by ranking the data first, then computing Pearson's r on the ranks.

| Feature | Pearson r | Spearman ρ |
|---------|-----------|------------|
| Measures | Linear association | Monotonic association |
| Data type | Continuous | Continuous or ordinal |
| Sensitive to outliers | Yes | No (uses ranks) |
| Assumption | Normal-ish | None (non-parametric) |

**When to use Spearman?**
- Ordinal data (survey ratings, rankings)
- Non-linear but monotonic relationships
- Data with outliers
- Non-normal distributions

Compare Pearson and Spearman correlations on data with a monotonic but non-linear relationship, and on data with outliers.

In [ ]:
np.random.seed(42)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Monotonic non-linear: exponential relationship
x1 = np.random.uniform(0, 5, 80)
y1 = np.exp(0.5 * x1) + np.random.normal(0, 0.5, 80)
r_pear, _ = stats.pearsonr(x1, y1)
r_spear, _ = stats.spearmanr(x1, y1)
axes[0].scatter(x1, y1, alpha=0.6, color='steelblue')
axes[0].set_title(f'Monotonic Non-Linear\nPearson={r_pear:.3f}, Spearman={r_spear:.3f}', fontsize=12)

# 2. Linear with outliers
x2 = np.random.normal(5, 2, 80)
y2 = 2 * x2 + np.random.normal(0, 2, 80)
# Add outliers
x2 = np.append(x2, [15, -5, 14])
y2 = np.append(y2, [-10, 25, -8])
r_pear2, _ = stats.pearsonr(x2, y2)
r_spear2, _ = stats.spearmanr(x2, y2)
axes[1].scatter(x2[:-3], y2[:-3], alpha=0.6, color='steelblue', label='Normal')
axes[1].scatter(x2[-3:], y2[-3:], alpha=0.9, color='red', s=100, marker='x', 
                linewidths=3, label='Outliers')
axes[1].set_title(f'Linear + Outliers\nPearson={r_pear2:.3f}, Spearman={r_spear2:.3f}', fontsize=12)
axes[1].legend(fontsize=10)

# 3. Ordinal data: course ratings
x3 = np.random.choice([1, 2, 3, 4, 5], size=100, p=[0.05, 0.15, 0.30, 0.30, 0.20])
y3 = np.random.choice([1, 2, 3, 4, 5], size=100, 
                       p=[0.05, 0.10, 0.25, 0.35, 0.25])  # Correlated ordinal
# Add some correlation
mask = np.random.random(100) < 0.6
y3[mask] = np.clip(x3[mask] + np.random.choice([-1, 0, 0, 1], size=mask.sum()), 1, 5)
r_pear3, _ = stats.pearsonr(x3, y3)
r_spear3, _ = stats.spearmanr(x3, y3)
axes[2].scatter(x3 + np.random.normal(0, 0.1, 100), 
                y3 + np.random.normal(0, 0.1, 100),  # Jitter for visibility
                alpha=0.4, color='steelblue')
axes[2].set_title(f'Ordinal Data\nPearson={r_pear3:.3f}, Spearman={r_spear3:.3f}', fontsize=12)
axes[2].set_xticks([1, 2, 3, 4, 5])
axes[2].set_yticks([1, 2, 3, 4, 5])

for ax in axes:
    ax.set_xlabel('X', fontsize=11)
    ax.set_ylabel('Y', fontsize=11)

plt.suptitle('Pearson vs Spearman: When They Differ', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('Key observations:')
print('1. Non-linear monotonic: Spearman captures the relationship better')
print('2. Outliers: Spearman is more robust (less affected)')
print('3. Ordinal data: Spearman is the correct choice')

Compute Spearman correlation with significance test using scipy.

In [ ]:
# Spearman correlation with scipy
rho, p_val = stats.spearmanr(study_hours, exam_scores)
print(f'Study Hours vs Exam Scores:')
print(f'  Spearman ρ = {rho:.4f}, p = {p_val:.6f}')

# Kendall's τ: another rank-based measure (more conservative)
tau, p_tau = stats.kendalltau(study_hours, exam_scores)
print(f'  Kendall τ  = {tau:.4f}, p = {p_tau:.6f}')

# Correlation matrix with Spearman (Pandas)
print('\nSpearman Correlation Matrix:')
print(students.corr(method='spearman').round(3))

---
## 4. Correlation Pitfalls

### 4.1 Correlation ≠ Causation

The most important rule in statistics. A correlation between X and Y could mean:
- X causes Y
- Y causes X
- A third variable Z causes both (confounding)
- Pure coincidence (spurious correlation)

### 4.2 Other Pitfalls
- **Restricted range**: If you only measure extreme values, correlation appears weaker
- **Outliers**: A single outlier can dramatically change r
- **Non-linearity**: Pearson r can be zero even with a strong non-linear relationship
- **Simpson's paradox**: Trend reverses when groups are combined

Demonstrate the confounding variable problem: ice cream sales and drowning deaths are correlated, but only because both are caused by temperature.

In [ ]:
# Confounding variable example
np.random.seed(42)
n = 100

# Temperature is the CONFOUND
temperature = np.random.normal(25, 8, n)  # Celsius
ice_cream_sales = 50 + 3 * temperature + np.random.normal(0, 10, n)  # Caused by temperature
drownings = 2 + 0.15 * temperature + np.random.normal(0, 1.5, n)    # Also caused by temperature

# Ice cream and drownings are correlated but NOT causally related!
r_spurious, p_spurious = stats.pearsonr(ice_cream_sales, drownings)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Spurious correlation
axes[0].scatter(ice_cream_sales, drownings, alpha=0.5, color='coral')
axes[0].set_xlabel('Ice Cream Sales', fontsize=12)
axes[0].set_ylabel('Drowning Deaths', fontsize=12)
axes[0].set_title(f'Spurious Correlation!\nr = {r_spurious:.3f}, p = {p_spurious:.4f}', fontsize=13)

# The real causes
r1, _ = stats.pearsonr(temperature, ice_cream_sales)
axes[1].scatter(temperature, ice_cream_sales, alpha=0.5, color='steelblue')
axes[1].set_xlabel('Temperature (°C)', fontsize=12)
axes[1].set_ylabel('Ice Cream Sales', fontsize=12)
axes[1].set_title(f'Real Cause: Temp → Sales\nr = {r1:.3f}', fontsize=13)

r2, _ = stats.pearsonr(temperature, drownings)
axes[2].scatter(temperature, drownings, alpha=0.5, color='seagreen')
axes[2].set_xlabel('Temperature (°C)', fontsize=12)
axes[2].set_ylabel('Drowning Deaths', fontsize=12)
axes[2].set_title(f'Real Cause: Temp → Drownings\nr = {r2:.3f}', fontsize=13)

plt.suptitle('Correlation ≠ Causation: The Confounding Variable', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

Demonstrate Simpson's Paradox: a trend that appears in several groups reverses when the groups are combined.

In [ ]:
# Simpson's Paradox
np.random.seed(42)

# Three departments with POSITIVE correlation within each
groups = {'Engineering': (80, 85, 40), 'Business': (60, 65, 50), 'Arts': (40, 45, 35)}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

all_x, all_y = [], []
colors = {'Engineering': 'steelblue', 'Business': 'coral', 'Arts': 'seagreen'}

for name, (cx, cy, n) in groups.items():
    x = np.random.normal(cx, 8, n)
    y = 0.3 * (x - cx) + cy + np.random.normal(0, 3, n)  # Positive within group
    all_x.extend(x)
    all_y.extend(y)
    
    # Plot by group
    axes[0].scatter(x, y, alpha=0.6, color=colors[name], label=name, s=40)
    # Regression line per group
    z = np.polyfit(x, y, 1)
    x_line = np.linspace(x.min(), x.max(), 50)
    axes[0].plot(x_line, np.polyval(z, x_line), color=colors[name], linewidth=2)

axes[0].set_title('Within Groups: POSITIVE correlation', fontsize=13)
axes[0].set_xlabel('Study Hours', fontsize=12)
axes[0].set_ylabel('Exam Score', fontsize=12)
axes[0].legend(fontsize=10)

# Combined: overall NEGATIVE correlation!
all_x, all_y = np.array(all_x), np.array(all_y)
axes[1].scatter(all_x, all_y, alpha=0.4, color='gray', s=40)
z_all = np.polyfit(all_x, all_y, 1)
x_line = np.linspace(all_x.min(), all_x.max(), 100)
axes[1].plot(x_line, np.polyval(z_all, x_line), 'r-', linewidth=2.5)
r_overall, _ = stats.pearsonr(all_x, all_y)
axes[1].set_title(f'Combined Data: NEGATIVE correlation (r = {r_overall:.3f})', fontsize=13)
axes[1].set_xlabel('Study Hours', fontsize=12)
axes[1].set_ylabel('Exam Score', fontsize=12)

plt.suptitle("Simpson's Paradox: Trend Reverses When Groups are Combined!", 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 5. Simple Linear Regression

**Linear regression** models the relationship between a dependent variable (y) and one independent variable (x):

$$y = \beta_0 + \beta_1 x + \epsilon$$

Where:
- **β₀** (intercept): predicted y when x = 0
- **β₁** (slope): change in y for a one-unit increase in x
- **ε** (error): random noise ~ N(0, σ²)

The **Ordinary Least Squares (OLS)** method finds β₀ and β₁ by minimizing the sum of squared residuals:
$$\min \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

### Key Metrics

| Metric | Formula | Interpretation |
|--------|---------|---------------|
| **R²** | 1 - SS_res/SS_total | Proportion of variance explained (0 to 1) |
| **RMSE** | √(mean(residuals²)) | Average prediction error (same units as y) |
| **MAE** | mean(\|residuals\|) | Average absolute error |

### 5.1 Fitting a Simple Regression by Hand

Understand OLS by computing the slope and intercept from scratch.

Compute the regression coefficients manually using the OLS formulas, then verify with NumPy and SciPy.

In [ ]:
# Manual OLS computation
x = study_hours
y = exam_scores

# Slope: β₁ = Σ(xᵢ - x̄)(yᵢ - ȳ) / Σ(xᵢ - x̄)²
beta_1 = np.sum((x - x.mean()) * (y - y.mean())) / np.sum((x - x.mean())**2)

# Intercept: β₀ = ȳ - β₁·x̄
beta_0 = y.mean() - beta_1 * x.mean()

# Predictions
y_pred = beta_0 + beta_1 * x

# R² = 1 - SS_res / SS_total
ss_res = np.sum((y - y_pred)**2)
ss_total = np.sum((y - y.mean())**2)
r_squared = 1 - ss_res / ss_total

# RMSE and MAE
rmse = np.sqrt(np.mean((y - y_pred)**2))
mae = np.mean(np.abs(y - y_pred))

print(f'=== Manual OLS ===')
print(f'ŷ = {beta_0:.2f} + {beta_1:.2f}·x')
print(f'\nInterpretation:')
print(f'  Intercept: Expected score with 0 study hours = {beta_0:.1f}')
print(f'  Slope: Each additional hour → +{beta_1:.2f} points')
print(f'\nR² = {r_squared:.4f} ({r_squared*100:.1f}% of variance explained)')
print(f'RMSE = {rmse:.2f} points')
print(f'MAE = {mae:.2f} points')

# Verify with scipy
slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)
print(f'\n=== SciPy linregress ===')
print(f'ŷ = {intercept:.2f} + {slope:.2f}·x')
print(f'R² = {r_value**2:.4f}, p = {p_value:.6f}')

Visualize the regression line, residuals, and the concept of minimizing squared errors.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Regression line with residuals shown
ax = axes[0]
ax.scatter(x, y, alpha=0.6, color='steelblue', s=50, label='Data')
x_sorted = np.sort(x)
ax.plot(x_sorted, beta_0 + beta_1 * x_sorted, 'r-', linewidth=2.5, label=f'ŷ = {beta_0:.1f} + {beta_1:.2f}x')

# Show residuals as vertical lines
for xi, yi, yp in zip(x, y, y_pred):
    ax.plot([xi, xi], [yi, yp], color='gray', linewidth=0.8, alpha=0.5)

ax.set_xlabel('Study Hours', fontsize=12)
ax.set_ylabel('Exam Score', fontsize=12)
ax.set_title(f'Simple Linear Regression (R² = {r_squared:.3f})', fontsize=13)
ax.legend(fontsize=11)

# 2. Actual vs Predicted
ax = axes[1]
ax.scatter(y, y_pred, alpha=0.6, color='steelblue', s=50)
lims = [min(y.min(), y_pred.min()) - 5, max(y.max(), y_pred.max()) + 5]
ax.plot(lims, lims, 'r--', linewidth=2, label='Perfect prediction')
ax.set_xlabel('Actual Score', fontsize=12)
ax.set_ylabel('Predicted Score', fontsize=12)
ax.set_title('Actual vs Predicted', fontsize=13)
ax.legend(fontsize=11)
ax.set_aspect('equal')

plt.tight_layout()
plt.show()

---
## 6. Regression with statsmodels

`statsmodels` is Python's primary library for statistical modeling. Its `OLS` function provides a comprehensive summary including:
- Coefficient estimates with standard errors and p-values
- R² and adjusted R²
- F-statistic for overall model significance
- Confidence intervals for each coefficient

Two interfaces:
1. **`sm.OLS(y, X)`**: Matrix-based (need to add constant manually)
2. **`smf.ols('y ~ x', data=df)`**: Formula-based (R-style, adds constant automatically)

### 6.1 The OLS Summary

Fit a regression model using both statsmodels interfaces and read the detailed output.

Fit a simple linear regression with statsmodels and interpret the key components of the summary output.

In [ ]:
# Method 1: Formula interface (recommended — easier and cleaner)
model_formula = smf.ols('score ~ hours', data=df).fit()
print(model_formula.summary())

Extract and explain the key quantities from the regression output.

In [ ]:
# Method 2: Matrix interface (need to add constant)
X = sm.add_constant(study_hours)  # Adds column of 1s for intercept
model_matrix = sm.OLS(exam_scores, X).fit()

# Key results extraction
print('=== Key Results ===')
print(f'Intercept (β₀): {model_formula.params["Intercept"]:.4f}')
print(f'Slope (β₁):     {model_formula.params["hours"]:.4f}')
print(f'\nR²:          {model_formula.rsquared:.4f}')
print(f'Adj. R²:     {model_formula.rsquared_adj:.4f}')
print(f'F-statistic: {model_formula.fvalue:.4f} (p = {model_formula.f_pvalue:.6f})')
print(f'\nCoefficient p-values:')
print(f'  Intercept: p = {model_formula.pvalues["Intercept"]:.6f}')
print(f'  Hours:     p = {model_formula.pvalues["hours"]:.6f}')
print(f'\n95% Confidence Intervals:')
print(model_formula.conf_int().round(4))

### 6.2 Making Predictions

Use the fitted model to predict exam scores for new study hour values.

Generate predictions with confidence and prediction intervals for new data points.

In [ ]:
# Point predictions
new_hours = pd.DataFrame({'hours': [2, 5, 8, 12]})
predictions = model_formula.predict(new_hours)

print('Point Predictions:')
for h, p in zip(new_hours['hours'], predictions):
    print(f'  {h} hours → predicted score = {p:.1f}')

# Prediction intervals (wider than CI — accounts for individual variation)
pred_summary = model_formula.get_prediction(new_hours).summary_frame(alpha=0.05)
print(f'\nPredictions with 95% Intervals:')
print(pred_summary[['mean', 'mean_ci_lower', 'mean_ci_upper', 
                     'obs_ci_lower', 'obs_ci_upper']].round(2).to_string())
print('\nmean_ci = confidence interval for the MEAN response')
print('obs_ci  = prediction interval for a NEW observation (wider)')

Visualize the regression with both confidence and prediction bands.

In [ ]:
# Regression with confidence and prediction bands
x_grid = pd.DataFrame({'hours': np.linspace(0, 12, 100)})
pred_all = model_formula.get_prediction(x_grid).summary_frame(alpha=0.05)

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(study_hours, exam_scores, alpha=0.5, color='steelblue', s=50, label='Data')
ax.plot(x_grid['hours'], pred_all['mean'], 'r-', linewidth=2.5, label='Regression line')
ax.fill_between(x_grid['hours'], pred_all['mean_ci_lower'], pred_all['mean_ci_upper'],
                alpha=0.3, color='red', label='95% CI (mean)')
ax.fill_between(x_grid['hours'], pred_all['obs_ci_lower'], pred_all['obs_ci_upper'],
                alpha=0.1, color='blue', label='95% PI (individual)')

ax.set_xlabel('Study Hours', fontsize=12)
ax.set_ylabel('Exam Score', fontsize=12)
ax.set_title('Regression with Confidence and Prediction Intervals', fontsize=14)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

---
## 7. Regression Diagnostics

A regression model is only valid if its **assumptions** hold. The four key assumptions (LINE):

| Assumption | Check | Problem |
|------------|-------|---------|
| **L**inearity | Residual vs fitted plot | Non-linear pattern |
| **I**ndependence | Durbin-Watson test | Autocorrelation |
| **N**ormality of residuals | Q-Q plot, Shapiro-Wilk | Heavy tails, skew |
| **E**qual variance (homoscedasticity) | Residual vs fitted plot | Fan/cone shape |

We check these by analyzing the **residuals** (e = y - ŷ).

Create a comprehensive 4-panel diagnostic plot for our regression model.

In [ ]:
def regression_diagnostics(model, figsize=(14, 10)):
    """Create 4-panel regression diagnostic plots."""
    residuals = model.resid
    fitted = model.fittedvalues
    std_resid = model.get_influence().resid_studentized_internal
    
    fig, axes = plt.subplots(2, 2, figsize=figsize)
    
    # 1. Residuals vs Fitted (check linearity + homoscedasticity)
    ax = axes[0, 0]
    ax.scatter(fitted, residuals, alpha=0.5, color='steelblue')
    ax.axhline(0, color='red', linestyle='--', linewidth=2)
    # LOWESS smoothing line
    lowess = sm.nonparametric.lowess(residuals, fitted, frac=0.6)
    ax.plot(lowess[:, 0], lowess[:, 1], 'orange', linewidth=2)
    ax.set_xlabel('Fitted Values', fontsize=11)
    ax.set_ylabel('Residuals', fontsize=11)
    ax.set_title('1. Residuals vs Fitted\n(Check: linearity, homoscedasticity)', fontsize=12)
    
    # 2. Q-Q Plot (check normality)
    ax = axes[0, 1]
    stats.probplot(std_resid, dist='norm', plot=ax)
    ax.set_title('2. Q-Q Plot\n(Check: normality of residuals)', fontsize=12)
    
    # 3. Scale-Location (check homoscedasticity)
    ax = axes[1, 0]
    ax.scatter(fitted, np.sqrt(np.abs(std_resid)), alpha=0.5, color='steelblue')
    lowess2 = sm.nonparametric.lowess(np.sqrt(np.abs(std_resid)), fitted, frac=0.6)
    ax.plot(lowess2[:, 0], lowess2[:, 1], 'orange', linewidth=2)
    ax.set_xlabel('Fitted Values', fontsize=11)
    ax.set_ylabel('√|Standardized Residuals|', fontsize=11)
    ax.set_title('3. Scale-Location\n(Check: equal variance)', fontsize=12)
    
    # 4. Residuals histogram
    ax = axes[1, 1]
    ax.hist(residuals, bins=20, density=True, color='steelblue', alpha=0.7, edgecolor='white')
    x_range = np.linspace(residuals.min(), residuals.max(), 100)
    ax.plot(x_range, stats.norm.pdf(x_range, residuals.mean(), residuals.std()), 
            'r-', linewidth=2)
    ax.set_xlabel('Residuals', fontsize=11)
    ax.set_ylabel('Density', fontsize=11)
    ax.set_title('4. Residual Distribution\n(Check: normality)', fontsize=12)
    
    plt.tight_layout()
    plt.show()
    
    # Numerical tests
    print('=== Numerical Diagnostic Tests ===')
    # Shapiro-Wilk for normality
    shapiro_stat, shapiro_p = stats.shapiro(residuals)
    print(f'Shapiro-Wilk (normality): W = {shapiro_stat:.4f}, p = {shapiro_p:.4f} '
          f'→ {"Normal" if shapiro_p > 0.05 else "Non-normal"}')
    
    # Durbin-Watson for independence
    from statsmodels.stats.stattools import durbin_watson
    dw = durbin_watson(residuals)
    print(f'Durbin-Watson (independence): DW = {dw:.4f} '
          f'(≈2 is good; <1.5 or >2.5 suggests autocorrelation)')
    
    # Breusch-Pagan for homoscedasticity
    from statsmodels.stats.diagnostic import het_breuschpagan
    bp_stat, bp_p, _, _ = het_breuschpagan(residuals, model.model.exog)
    print(f'Breusch-Pagan (homoscedasticity): LM = {bp_stat:.4f}, p = {bp_p:.4f} '
          f'→ {"Homoscedastic" if bp_p > 0.05 else "Heteroscedastic"}')

# Run diagnostics on our model
regression_diagnostics(model_formula)

### 7.1 What Bad Diagnostics Look Like

Let's create examples of violated assumptions so you can recognize them.

Generate three datasets that violate different regression assumptions and show their diagnostic plots.

In [ ]:
np.random.seed(42)
n = 100
x_demo = np.random.uniform(1, 10, n)

fig, axes = plt.subplots(3, 2, figsize=(14, 12))

# 1. Non-linearity: quadratic relationship fitted with linear model
y_nonlin = 5 + 2*x_demo + 0.5*x_demo**2 + np.random.normal(0, 3, n)
model_nl = smf.ols('y ~ x', data=pd.DataFrame({'x': x_demo, 'y': y_nonlin})).fit()
axes[0, 0].scatter(model_nl.fittedvalues, model_nl.resid, alpha=0.5, color='coral')
axes[0, 0].axhline(0, color='red', linestyle='--')
lowess_nl = sm.nonparametric.lowess(model_nl.resid, model_nl.fittedvalues, frac=0.6)
axes[0, 0].plot(lowess_nl[:, 0], lowess_nl[:, 1], 'k-', linewidth=2)
axes[0, 0].set_title('NON-LINEAR: Curved pattern in residuals', fontsize=12, color='red')
axes[0, 0].set_xlabel('Fitted Values')
axes[0, 0].set_ylabel('Residuals')

stats.probplot(model_nl.resid, plot=axes[0, 1])
axes[0, 1].set_title('Non-linearity: Q-Q Plot', fontsize=12)

# 2. Heteroscedasticity: variance increases with x
y_hetero = 3 + 2*x_demo + np.random.normal(0, 1, n) * x_demo  # Var increases with x
model_het = smf.ols('y ~ x', data=pd.DataFrame({'x': x_demo, 'y': y_hetero})).fit()
axes[1, 0].scatter(model_het.fittedvalues, model_het.resid, alpha=0.5, color='coral')
axes[1, 0].axhline(0, color='red', linestyle='--')
axes[1, 0].set_title('HETEROSCEDASTIC: Fan/cone shape', fontsize=12, color='red')
axes[1, 0].set_xlabel('Fitted Values')
axes[1, 0].set_ylabel('Residuals')

axes[1, 1].scatter(model_het.fittedvalues, 
                   np.sqrt(np.abs(model_het.get_influence().resid_studentized_internal)),
                   alpha=0.5, color='coral')
axes[1, 1].set_title('Heteroscedastic: Scale-Location', fontsize=12)
axes[1, 1].set_xlabel('Fitted Values')
axes[1, 1].set_ylabel('√|Std Residuals|')

# 3. Non-normal residuals: heavy-tailed errors
y_nonnorm = 3 + 2*x_demo + np.random.standard_t(3, n) * 3  # t-distribution errors
model_nn = smf.ols('y ~ x', data=pd.DataFrame({'x': x_demo, 'y': y_nonnorm})).fit()
axes[2, 0].hist(model_nn.resid, bins=25, density=True, color='coral', alpha=0.7, edgecolor='white')
x_norm = np.linspace(model_nn.resid.min(), model_nn.resid.max(), 100)
axes[2, 0].plot(x_norm, stats.norm.pdf(x_norm, 0, model_nn.resid.std()), 'k-', linewidth=2)
axes[2, 0].set_title('NON-NORMAL: Heavy tails', fontsize=12, color='red')
axes[2, 0].set_xlabel('Residuals')

stats.probplot(model_nn.resid, plot=axes[2, 1])
axes[2, 1].set_title('Non-normal: Deviations at tails in Q-Q', fontsize=12, color='red')

fig.suptitle('Recognizing Violated Assumptions', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
## 8. Multiple Linear Regression

**Multiple regression** extends the model to include several predictors:

$$y = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \cdots + \beta_p x_p + \epsilon$$

Each coefficient βⱼ represents the change in y for a one-unit increase in xⱼ, **holding all other predictors constant** (partial effect).

**Why multiple regression?**
- Control for confounding variables
- Better predictions using multiple inputs
- Understand which factors matter most

### 8.1 Building a Multiple Regression Model

Predict house prices using multiple features: size, bedrooms, age, and distance to city center.

Generate a realistic housing dataset and fit a multiple regression model.

In [ ]:
# Generate housing data
np.random.seed(42)
n = 200

size_sqft = np.random.normal(1500, 400, n).clip(600, 4000)
bedrooms = np.random.choice([1, 2, 3, 4, 5], size=n, p=[0.05, 0.20, 0.40, 0.25, 0.10])
age_years = np.random.uniform(0, 50, n)
dist_city = np.random.exponential(5, n).clip(0.5, 30)

# True model: price depends on all four
price = (50000 + 120 * size_sqft + 15000 * bedrooms - 800 * age_years 
         - 3000 * dist_city + np.random.normal(0, 30000, n))
price = price.clip(50000)  # No negative prices

houses = pd.DataFrame({
    'price': price,
    'size': size_sqft,
    'bedrooms': bedrooms,
    'age': age_years,
    'distance': dist_city
})

print('Housing Data:')
print(houses.describe().round(1))

Fit the multiple regression model and interpret the coefficients.

In [ ]:
# Fit multiple regression
model_multi = smf.ols('price ~ size + bedrooms + age + distance', data=houses).fit()
print(model_multi.summary())

Extract and clearly interpret each coefficient in plain language.

In [ ]:
# Interpret coefficients
print('=== Coefficient Interpretation ===')
print(f'Intercept: ${model_multi.params["Intercept"]:,.0f}')
print(f'  → Base price when all features are zero\n')

for var in ['size', 'bedrooms', 'age', 'distance']:
    coef = model_multi.params[var]
    pval = model_multi.pvalues[var]
    sig = '*' if pval < 0.05 else ''
    print(f'{var:>10}: ${coef:>10,.1f}  (p={pval:.4f}) {sig}')

print(f'\nInterpretation (holding other variables constant):')
print(f'  +1 sqft       → +${model_multi.params["size"]:,.0f} in price')
print(f'  +1 bedroom    → +${model_multi.params["bedrooms"]:,.0f} in price')
print(f'  +1 year older → ${model_multi.params["age"]:,.0f} in price')
print(f'  +1 km further → ${model_multi.params["distance"]:,.0f} in price')
print(f'\nR² = {model_multi.rsquared:.4f}')
print(f'Adj. R² = {model_multi.rsquared_adj:.4f}')

### 8.2 Multicollinearity: VIF

**Multicollinearity** occurs when predictors are highly correlated with each other. This inflates standard errors and makes individual coefficients unreliable.

The **Variance Inflation Factor (VIF)** measures how much the variance of each coefficient is inflated:
- VIF = 1: No correlation with other predictors
- VIF > 5: Moderate concern
- VIF > 10: Serious multicollinearity

Compute VIF for each predictor to check for multicollinearity in our housing model.

In [ ]:
# Compute VIF for each predictor
X = houses[['size', 'bedrooms', 'age', 'distance']]
X_with_const = sm.add_constant(X)

vif_data = pd.DataFrame()
vif_data['Variable'] = X_with_const.columns
vif_data['VIF'] = [variance_inflation_factor(X_with_const.values, i) 
                    for i in range(X_with_const.shape[1])]

# Skip the constant
vif_data = vif_data[vif_data['Variable'] != 'const']

print('Variance Inflation Factors:')
print(vif_data.to_string(index=False))
print(f'\nAll VIF < 5: No multicollinearity concerns.')

# Show predictor correlations
print('\nPredictor Correlation Matrix:')
print(X.corr().round(3))

---
## 9. Categorical Predictors (Dummy Variables)

Categorical variables (like region, color, or type) can't be used directly in regression. We convert them to **dummy variables** (0/1 indicators).

A category with k levels needs **k-1 dummy variables** (one is the reference/baseline).

In the `statsmodels` formula interface, use `C(variable)` to automatically create dummy variables.

Add a categorical variable (neighborhood) to our housing model and interpret the dummy variable coefficients.

In [ ]:
# Add neighborhood as a categorical predictor
np.random.seed(42)
neighborhoods = np.random.choice(['Downtown', 'Suburb', 'Rural'], size=n, p=[0.3, 0.5, 0.2])
# Neighborhood affects price
hood_effect = np.where(neighborhoods == 'Downtown', 40000, 
                        np.where(neighborhoods == 'Suburb', 15000, 0))
houses['neighborhood'] = neighborhoods
houses['price'] = houses['price'] + hood_effect

# Fit model with categorical variable
model_cat = smf.ols('price ~ size + bedrooms + age + distance + C(neighborhood)', 
                     data=houses).fit()
print(model_cat.summary())

Explain how to interpret dummy variable coefficients — each represents the difference from the reference category.

In [ ]:
# Interpret categorical coefficients
print('=== Categorical Variable Interpretation ===')
print(f'Reference category: Downtown (baseline, coefficient = 0)')
print(f'\nCompared to Downtown:')
for param_name, coef in model_cat.params.items():
    if 'neighborhood' in param_name:
        category = param_name.split('[T.')[1].rstrip(']')
        pval = model_cat.pvalues[param_name]
        print(f'  {category}: {"$" + f"{coef:+,.0f}"} (p = {pval:.4f})')

print(f'\nR² improved from {model_multi.rsquared:.4f} to {model_cat.rsquared:.4f} '
      f'by adding neighborhood')

# Manual dummy coding example
print('\n=== How Dummy Variables Work ===')
dummies = pd.get_dummies(houses['neighborhood'], drop_first=True, dtype=int)
print(houses[['neighborhood']].join(dummies).head(10))

---
## 10. Model Comparison

How do we choose between models? We need metrics that balance **fit** with **complexity**.

| Metric | Formula Idea | When to Use |
|--------|-------------|-------------|
| **R²** | Variance explained | Never for model selection alone (always increases with more vars) |
| **Adjusted R²** | R² penalized for # predictors | Comparing nested models |
| **AIC** | -2·loglik + 2·k | Comparing any models (lower is better) |
| **BIC** | -2·loglik + k·ln(n) | Comparing any models, penalizes complexity more |

Compare multiple models of increasing complexity to find the best balance of fit and parsimony.

In [ ]:
# Compare multiple models
models = {
    'Size only': smf.ols('price ~ size', data=houses).fit(),
    'Size + Beds': smf.ols('price ~ size + bedrooms', data=houses).fit(),
    'Size + Beds + Age': smf.ols('price ~ size + bedrooms + age', data=houses).fit(),
    'All numeric': smf.ols('price ~ size + bedrooms + age + distance', data=houses).fit(),
    'All + Neighborhood': model_cat,
}

comparison = pd.DataFrame({
    'Model': models.keys(),
    'Variables': [1, 2, 3, 4, 6],
    'R²': [m.rsquared for m in models.values()],
    'Adj. R²': [m.rsquared_adj for m in models.values()],
    'AIC': [m.aic for m in models.values()],
    'BIC': [m.bic for m in models.values()],
    'RMSE': [np.sqrt(m.mse_resid) for m in models.values()]
}).round(4)

print('=== Model Comparison ===')
print(comparison.to_string(index=False))

best_aic = comparison.loc[comparison['AIC'].idxmin(), 'Model']
best_bic = comparison.loc[comparison['BIC'].idxmin(), 'Model']
print(f'\nBest by AIC: {best_aic}')
print(f'Best by BIC: {best_bic}')

Visualize how model metrics change as we add more predictors.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

model_names = list(models.keys())
x_pos = range(len(model_names))

# R² and Adjusted R²
axes[0].plot(x_pos, comparison['R²'], 'bo-', markersize=8, linewidth=2, label='R²')
axes[0].plot(x_pos, comparison['Adj. R²'], 'rs-', markersize=8, linewidth=2, label='Adj. R²')
axes[0].set_ylabel('Score', fontsize=12)
axes[0].set_title('R² vs Adjusted R²', fontsize=13)
axes[0].legend(fontsize=11)
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels([f'M{i+1}' for i in x_pos], fontsize=10)

# AIC and BIC
axes[1].plot(x_pos, comparison['AIC'], 'bo-', markersize=8, linewidth=2, label='AIC')
axes[1].plot(x_pos, comparison['BIC'], 'rs-', markersize=8, linewidth=2, label='BIC')
axes[1].set_ylabel('Information Criterion', fontsize=12)
axes[1].set_title('AIC & BIC (Lower = Better)', fontsize=13)
axes[1].legend(fontsize=11)
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels([f'M{i+1}' for i in x_pos], fontsize=10)

# RMSE
axes[2].bar(x_pos, comparison['RMSE'], color='seagreen', alpha=0.7)
axes[2].set_ylabel('RMSE ($)', fontsize=12)
axes[2].set_title('Root Mean Square Error', fontsize=13)
axes[2].set_xticks(x_pos)
axes[2].set_xticklabels([f'M{i+1}' for i in x_pos], fontsize=10)

plt.suptitle('Model Comparison: M1=Size, M2=+Beds, M3=+Age, M4=+Dist, M5=+Neighborhood', 
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

---
## 11. Practical Example: Predicting Employee Salary

A complete regression analysis workflow: from exploratory analysis to model building and interpretation.

### Step 1: Create and Explore the Dataset

Generate a realistic employee salary dataset and perform initial exploration.

In [ ]:
np.random.seed(42)
n = 300

experience = np.random.exponential(7, n).clip(0.5, 35).round(1)
education = np.random.choice(['Bachelor', 'Master', 'PhD'], size=n, p=[0.50, 0.35, 0.15])
department = np.random.choice(['Engineering', 'Marketing', 'Sales', 'HR'], size=n, 
                               p=[0.30, 0.25, 0.30, 0.15])
performance = np.random.normal(3.5, 0.8, n).clip(1, 5).round(1)

# True salary model
edu_bonus = np.where(education == 'PhD', 18000, np.where(education == 'Master', 8000, 0))
dept_bonus = np.where(department == 'Engineering', 12000, 
                      np.where(department == 'Marketing', 3000,
                      np.where(department == 'Sales', 5000, 0)))

salary = (35000 + 2500 * experience + edu_bonus + dept_bonus 
          + 4000 * performance + np.random.normal(0, 8000, n)).clip(25000)

employees = pd.DataFrame({
    'salary': salary.round(0),
    'experience': experience,
    'education': education,
    'department': department,
    'performance': performance
})

print('Employee Dataset:')
print(employees.describe().round(1))
print(f'\nEducation: {employees["education"].value_counts().to_dict()}')
print(f'Department: {employees["department"].value_counts().to_dict()}')

### Step 2: Exploratory Visualization

Explore relationships between salary and each predictor before building the model.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Salary vs Experience
sns.scatterplot(data=employees, x='experience', y='salary', hue='education',
                alpha=0.5, ax=axes[0, 0])
axes[0, 0].set_title('Salary vs Experience (by Education)', fontsize=13)

# Salary by Education
sns.boxplot(data=employees, x='education', y='salary', 
            order=['Bachelor', 'Master', 'PhD'], ax=axes[0, 1],
            palette='Set2')
axes[0, 1].set_title('Salary by Education Level', fontsize=13)

# Salary by Department
sns.boxplot(data=employees, x='department', y='salary', ax=axes[1, 0],
            palette='Set3')
axes[1, 0].set_title('Salary by Department', fontsize=13)

# Salary vs Performance
sns.scatterplot(data=employees, x='performance', y='salary', alpha=0.4, ax=axes[1, 1])
axes[1, 1].set_title('Salary vs Performance Rating', fontsize=13)

# Add correlation
r_exp, _ = stats.pearsonr(employees['experience'], employees['salary'])
r_perf, _ = stats.pearsonr(employees['performance'], employees['salary'])
axes[0, 0].annotate(f'r = {r_exp:.3f}', xy=(0.05, 0.95), xycoords='axes fraction', fontsize=12)
axes[1, 1].annotate(f'r = {r_perf:.3f}', xy=(0.05, 0.95), xycoords='axes fraction', fontsize=12)

plt.tight_layout()
plt.show()

### Step 3: Build and Compare Models

Fit progressively more complex models and select the best one.

In [ ]:
# Model 1: Experience only
m1 = smf.ols('salary ~ experience', data=employees).fit()

# Model 2: + Education
m2 = smf.ols('salary ~ experience + C(education)', data=employees).fit()

# Model 3: + Department
m3 = smf.ols('salary ~ experience + C(education) + C(department)', data=employees).fit()

# Model 4: Full model (+ Performance)
m4 = smf.ols('salary ~ experience + C(education) + C(department) + performance', 
              data=employees).fit()

# Compare
model_comp = pd.DataFrame({
    'Model': ['Experience', '+ Education', '+ Department', '+ Performance'],
    'R²': [m.rsquared for m in [m1, m2, m3, m4]],
    'Adj R²': [m.rsquared_adj for m in [m1, m2, m3, m4]],
    'AIC': [m.aic for m in [m1, m2, m3, m4]],
    'BIC': [m.bic for m in [m1, m2, m3, m4]]
}).round(2)

print('Model Comparison:')
print(model_comp.to_string(index=False))

### Step 4: Interpret the Best Model

Print the full summary of the best model and provide plain-language interpretation.

In [ ]:
# Full model summary
print(m4.summary())

Create a coefficient plot to visualize the magnitude and significance of each predictor.

In [ ]:
# Coefficient plot (forest plot)
coef_df = pd.DataFrame({
    'Variable': m4.params.index[1:],  # Skip intercept
    'Coefficient': m4.params.values[1:],
    'CI_lower': m4.conf_int().iloc[1:, 0].values,
    'CI_upper': m4.conf_int().iloc[1:, 1].values,
    'p_value': m4.pvalues.values[1:]
})

# Clean up variable names
coef_df['Variable'] = coef_df['Variable'].str.replace('C\\(education\\)\\[T\\.', '', regex=True)
coef_df['Variable'] = coef_df['Variable'].str.replace('C\\(department\\)\\[T\\.', '', regex=True)
coef_df['Variable'] = coef_df['Variable'].str.rstrip(']')

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['steelblue' if p < 0.05 else 'gray' for p in coef_df['p_value']]
y_pos = range(len(coef_df))

ax.barh(y_pos, coef_df['Coefficient'], color=colors, alpha=0.7, height=0.6)
ax.errorbar(coef_df['Coefficient'], y_pos, 
            xerr=[coef_df['Coefficient'] - coef_df['CI_lower'], 
                  coef_df['CI_upper'] - coef_df['Coefficient']],
            fmt='none', color='black', capsize=3)
ax.axvline(0, color='red', linestyle='--', linewidth=1.5)
ax.set_yticks(y_pos)
ax.set_yticklabels(coef_df['Variable'], fontsize=11)
ax.set_xlabel('Coefficient Value ($)', fontsize=12)
ax.set_title('Regression Coefficients with 95% CI\n(blue = significant, gray = not)', fontsize=13)
plt.tight_layout()
plt.show()

### Step 5: Diagnostics and Final Report

Run diagnostic checks on the final model and compile a summary report.

In [ ]:
# Quick diagnostics
regression_diagnostics(m4)

Generate salary predictions for specific employee profiles.

In [ ]:
# Predict salary for specific profiles
profiles = pd.DataFrame({
    'experience': [2, 5, 10, 15],
    'education': ['Bachelor', 'Master', 'Master', 'PhD'],
    'department': ['Marketing', 'Engineering', 'Sales', 'Engineering'],
    'performance': [3.5, 4.0, 3.8, 4.5]
})

predictions = m4.get_prediction(profiles).summary_frame(alpha=0.05)

print('=== Salary Predictions ===')
for i, row in profiles.iterrows():
    pred = predictions.iloc[i]
    print(f'\nProfile: {row["experience"]}yr exp, {row["education"]}, '
          f'{row["department"]}, rating={row["performance"]}')
    print(f'  Predicted: ${pred["mean"]:,.0f}')
    print(f'  95% CI:    (${pred["mean_ci_lower"]:,.0f}, ${pred["mean_ci_upper"]:,.0f})')
    print(f'  95% PI:    (${pred["obs_ci_lower"]:,.0f}, ${pred["obs_ci_upper"]:,.0f})')

---
## 12. Summary

| Section | Key Concepts | Python Functions |
|---------|-------------|------------------|
| 1. Scatter Plots | Visualize before computing | `plt.scatter()`, `sns.regplot()` |
| 2. Pearson r | Linear correlation, -1 to +1 | `stats.pearsonr()`, `df.corr()` |
| 3. Spearman ρ | Rank-based, robust, monotonic | `stats.spearmanr()`, `df.corr('spearman')` |
| 4. Pitfalls | ≠ causation, outliers, Simpson's | Always visualize + think critically |
| 5. Simple Regression | ŷ = β₀ + β₁x, R², RMSE | `stats.linregress()`, `np.polyfit()` |
| 6. statsmodels OLS | Full inference, CI, prediction | `smf.ols('y ~ x', data).fit()` |
| 7. Diagnostics | LINE assumptions, residual plots | Custom function, `het_breuschpagan` |
| 8. Multiple Regression | Multiple predictors, partial effects | `smf.ols('y ~ x1 + x2', data)` |
| 9. Categorical Vars | Dummy coding, C() in formulas | `C(variable)`, `pd.get_dummies()` |
| 10. Model Comparison | Adj R², AIC, BIC | `model.aic`, `model.bic` |
| 11. Practical | End-to-end regression pipeline | All above combined |

### Key Takeaways

1. **Always visualize** before computing correlations or fitting models
2. **Correlation ≠ causation** — never forget confounders
3. **Use Spearman** for non-linear monotonic relationships or ordinal data
4. **Check assumptions** after fitting regression — use diagnostic plots
5. **Adjusted R² and AIC/BIC** are better than R² for model selection
6. **Interpret coefficients carefully** — "holding other variables constant"

---
## 13. Homework

### Task 1: Correlation Analysis
Use the `tips` dataset from seaborn (`sns.load_dataset('tips')`):
1. Compute the Pearson and Spearman correlations between `total_bill` and `tip`
2. Create a correlation heatmap for all numeric columns
3. Make a scatter plot of `total_bill` vs `tip`, colored by `time` (Lunch vs Dinner)
4. Is the correlation different for lunch vs dinner? Test each separately
5. Discuss: does a higher bill *cause* a higher tip, or could there be confounders?

### Task 2: Simple Linear Regression
Using the same `tips` dataset:
1. Fit a simple regression: `tip ~ total_bill`
2. Interpret the slope: how much does the tip increase per dollar of the bill?
3. What is the predicted tip for a $50 bill? Include 95% prediction interval
4. Create diagnostic plots — do the assumptions hold?
5. What is the R² and what does it mean in context?

### Task 3: Multiple Regression
Build a multiple regression model to predict `tip`:
1. Start with `tip ~ total_bill + size` (party size)
2. Add categorical variables: `tip ~ total_bill + size + C(sex) + C(smoker) + C(day) + C(time)`
3. Compare R², Adj R², AIC for the two models
4. Which predictors are significant? Which are not?
5. Check for multicollinearity using VIF
6. Create a coefficient plot for the full model

### Task 4: Real-World Regression Project
Create a dataset of 200 apartments with features: `area` (30-150 m²), `floor` (1-25), `rooms` (1-5), `age` (0-40 years), `has_parking` (0/1), `district` (A, B, C).
1. Define a true price model with realistic coefficients
2. Perform full EDA (scatter plots, correlations, box plots by district)
3. Build and compare at least 3 regression models of different complexity
4. Select the best model and run complete diagnostics
5. Predict prices for 5 specific apartment profiles with prediction intervals
6. Write a 3-sentence conclusion summarizing which factors most influence apartment prices

---
### Next Week Preview

**Week 10: ANOVA & Non-parametric Methods** — When we have **three or more groups**, t-tests aren't enough. We'll learn ANOVA (Analysis of Variance) for comparing multiple group means simultaneously, post-hoc tests for pairwise comparisons, and non-parametric alternatives (Mann-Whitney, Kruskal-Wallis) for when data violates normality assumptions.

---
*Applied Statistics with Python (2026) | Week 9 | Correlation & Regression*